In [14]:
import pandas as pd
import numpy as np

In [11]:
df = pd.read_csv('udemy_courses_with_student_ids_and_ratings.csv')
R = df.pivot(index='student_id',columns='course_id',values='course_rating')
R.head()


course_id,8324,8325,11153,11174,11475,12214,12975,13216,14571,15285,...,1273896,1274210,1274846,1275790,1275872,1276020,1276182,1276364,1277924,1282064
student_id,,,,,,,,,,,,,,,,,,,,,
STU00CN0KCP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
STU00MJMZ1J,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
STU013OQ715,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
STU01D55K29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
STU01FBPRI1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
mean = R.mean(axis=1)
R_new = R.sub(mean, axis=0).fillna(0)

In [ ]:
U, S, Vt = np.linalg.svd(R_new, full_matrices=False)

k = 2
U_k = U[:, :k]
S_k = np.diag(S[:k])
Vt_k = Vt[:k, :]

R_new1 = np.dot(np.dot(U_k,S_k),Vt_k)
R_new2 = pd.DataFrame(R_new1, index=R.index, columns=R.columns).add(mean,axis=0)

course_titles = (
    df[["course_id", "course_title"]].drop_duplicates().set_index("course_id")[
        "course_title"
    ]
)

(3678, 3672)


In [ ]:
def recommend_top_courses(student_id, top_n=3):

    student_scores = R_new2.loc[student_id]
    interacted_courses = R.loc[student_id].dropna().index
    unseen_scores = student_scores.drop(labels=interacted_courses)
    top_course_ids = unseen_scores.sort_values(ascending=False).head(top_n)

    recommendations = pd.DataFrame(
        {
            "Course ID": top_course_ids.index,
            "Course Title": top_course_ids.index.map(course_titles),
            "Predicted Rating": top_course_ids.values.round(2),
        }
    )

    return recommendations


recommendations = recommend_top_courses('STU01FBPRI1', top_n=3)
print(recommendations)

   Course ID                                 Course Title  Predicted Rating
0       8324                     Javascript for Beginners               4.3
1     864482   Coding Made Easy: HTML & CSS For Beginners               4.3
2     865630  Aprenda Cálculo Financeiro de Forma Simples               4.3
